In [1]:
import os
import pwd
import numpy as np
import pandas as pd
import sys

from pyspark.sql import SparkSession
from random import randrange
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.sql.window import Window
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

username = pwd.getpwuid(os.getuid()).pw_name
hadoopFS=os.getenv('HADOOP_FS', None)
groupName = "J1"

userns = f"iceberg.{username}_iceberg"
sharedns = 'iceberg.com490_iceberg'
groupfs = f"{hadoopFS}/user/groups/com-490/{groupName}"

print(os.getenv('SPARK_HOME'))
print(f"hadoopFSs={hadoopFS}")
print(f"userns={userns}")
print(f"groupfs={groupfs}")

/opt/spark
hadoopFSs=hdfs://iccluster061.iccluster.epfl.ch:9000
userns=iceberg.skalli_iceberg
groupfs=hdfs://iccluster061.iccluster.epfl.ch:9000/user/groups/com-490/J1


In [4]:
spark = (SparkSession\
            .builder
            .appName(username + '-assignment-2')
            .config('spark.ui.port', randrange(4050, 4450, 5))
            .config("spark.executorEnv.PYTHONPATH", ":".join(sys.path))
            .config('spark.jars',
                    f'{hadoopFS}/data/com-490/jars/iceberg-spark-runtime-3.5_2.13-1.6.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/sedona-spark-shaded-3.5_2.13-1.7.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/geotools-wrapper-1.7.1-28.5.jar'
            )
            .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
            .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
            .config('spark.sql.catalog.iceberg.type', 'hadoop')
            .config('spark.sql.catalog.iceberg.warehouse', f'{hadoopFS}/data/com-490/silver/')
            .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
            .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
            .config('spark.sql.catalog.spark_catalog.warehouse', f'{hadoopFS}/user/{username}/assignment-3/warehouse')
            .config("spark.sql.warehouse.dir", f'{hadoopFS}/user/{username}/assignment-3/spark/warehouse')
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.executor.instances", "4")
        ).master('yarn').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Exception in thread "main" java.nio.file.NoSuchFileException: /tmp/tmpm80r_xh1/connection2997529687479816128.info
	at java.base/sun.nio.fs.UnixException.translateToIOException(UnixException.java:92)
	at java.base/sun.nio.fs.UnixException.rethrowAsIOException(UnixException.java:106)
	at java.base/sun.nio.fs.UnixException.rethrowAsIOException(UnixException.java:111)
	at java.base/sun.nio.fs.UnixFileSystemProvider.newByteChannel(UnixFileSystemProvider.java:261)
	at java.base/java.nio.file.Files.newByteChannel(Files.java:380)
	at java.base/java.nio.file.Files.createFile(Files.java:658)
	at java.base/java.nio.file.TempFileHelper.create(TempFileHelper.java:136)
	at java.base/java.nio.file.TempFileHelper.createTempFile(TempFileHelper.java:159)
	at java.base/java.nio.file.Files.createTempFile(Files.java:879)
	at org.apache.spark.api.python.PythonGatewayServer$.main(

KeyboardInterrupt: 

In [ ]:
spark.sparkContext

In [4]:
# Load the full dataset you just saved
full_data_path = f"{hadoopFS}/user/{username}/full_sbb_prepared_dataset.parquet"
full_data = spark.read.parquet(full_data_path)

print(f"✅ Loaded Full Dataset from: {full_data_path}")

✅ Loaded Full Dataset from: hdfs://iccluster061.iccluster.epfl.ch:9000/user/skalli/full_sbb_prepared_dataset.parquet


# 🛠️ Robust Validation Pipeline: Multi-Tiered Test Suite Generation

This notebook implements a modular pipeline to create a **Ground Truth Benchmark**. By mining historical SBB `istdaten` (actual arrival/departure logs), we reconstruct real journeys that passengers took, including their actual delays. This allows us to test if our **CSA + ML** system can outperform a standard timetable-based search.

## 📋 Functional Architecture

### 1. `generate_test_suite()`: The Extraction Engine
This function is responsible for the heavy lifting. It performs multi-way joins in PySpark to find valid connections across the Swiss network.

#### **A. Sequence Identification (The How & Why)**
* **How**: We use PySpark Window functions to assign a `seq` (sequence number) to every stop within a `trip_id`, ordered by time.
* **Why**: This allows us to mathematically define the **Origin** (where `seq == 1`) and the **Destination** (where `seq == max_seq`) for any given train.

#### **B. Direct Journeys (Level 1 & 2)**
* **How**: We perform an inner join on the same `trip_id` between its start and end points.
* **Why**: These serve as the baseline. If a routing algorithm cannot solve a direct trip, it cannot solve a transfer.

#### **C. The Multi-Leg Transfer Logic (Level 3 - 8)**
* **How**: We define four "Legs" and join them at **Major Hubs** (e.g., Zürich HB, Bern). To be a valid transfer, the code enforces:
    * **Trip Change**: The `trip_id` of the incoming train must be different from the outgoing train.
    * **The Goldilocks Window**: The transfer must be at least **5 minutes** (minimum walking time) but no more than **30 minutes** (to avoid unreasonable waits).
* **Why**: Swiss transport follows a **"Hub-and-Spoke"** model. Restricting transfers to hubs mirrors real passenger behavior and prevents the "Combinatorial Explosion" that would crash the Spark cluster if we searched every tiny bus stop.

#### **D. Stratified Sampling (The "Even Distribution" Secret)**
* **How**: After gathering thousands of journeys, we use Pandas to group them by Level and take a random sample of `N` rows from each.
* **Why**: Real-world data is heavily biased toward "On-Time" and "Direct" trips. If we took a random sample, we would have 0 "Nightmare" cases. This step forces the dataset to be **perfectly balanced**, so the algorithm is tested equally on easy and extremely difficult cases.

---

## 🗂️ Data Lineage: Tables & Intermediate Structures

To compute complex 1, 2, and 3-transfer journeys, the pipeline breaks the SBB dataset into specialized intermediate tables (Legs). This "Divide and Conquer" approach is necessary to ensure the Spark Catalyst Optimizer can handle the multi-way joins without memory overflows.

| Table / Variable | Source | Role in Pipeline | Why? |
| :--- | :--- | :--- | :--- |
| `full_data` | SBB Ist-Daten | Primary Input | Contains the "Ground Truth" (Actual vs. Scheduled) for all Swiss trains, buses, and trams. |
| `leg1` | Derived from `day_data` | Origin Leg | Filters trips starting at any stop and ending at a Major Hub. Captures the first part of a journey. |
| `leg2` & `leg3` | Derived from `day_data` | Bridge Legs | Filters trips that start at one hub and end at another. Necessary for 2 and 3-transfer pathfinding. |
| `leg4` | Derived from `day_data` | Destination Leg | Filters trips starting at a hub and finishing at the final destination (`max_seq`). |
| `trans1, 2, 3` | Joined Legs | Journey Pools | Result of joining Legs together. For example, `trans2` is the result of `leg1 + leg2 + leg4`. |
| `ultimate_pool` | Union of Pools | Candidate Set | A unified Spark DataFrame containing all discovered paths before sampling. |

---

## 📉 The 8-Level Difficulty Matrix
We categorize every journey into one of eight tiers to evaluate the "Success Rate" of our algorithm across a spectrum of difficulty.

| Difficulty | Transfers | Delay Status | Purpose |
| :--- | :--- | :--- | :--- |
| **Level 1** | 0 | On-Time ($\leq$ 3m) | Baseline accuracy check. |
| **Level 2** | 0 | Delayed (> 3m) | Tests basic delay feature integration. |
| **Level 3** | 1 | On-Time ($\leq$ 3m) | Standard commute validation. |
| **Level 4** | 1 | Delayed (> 3m) | **Stress Test**: Can ML save a 1-transfer connection? |
| **Level 5** | 2 | On-Time ($\leq$ 3m) | Complex routing logic check. |
| **Level 6** | 2 | Delayed (> 3m) | **Nightmare**: Testing robustness in chaotic scenarios. |
| **Level 7** | 3 | On-Time ($\leq$ 3m) | Legendary complexity check. |
| **Level 8** | 3 | Delayed (> 3m) | **The Final Boss**: Maximum transfers + massive delays. |

---

### 2. `display_test_suite_stats()`: Quality Control
This function acts as a **"Sanity Check."** It displays the count of each level to ensure the Stratified Sampling worked correctly and provides a preview of the `transfer_hub` column (e.g., "Bern -> Olten -> Zürich HB") to verify the journey paths are logical.

### 3. `save_test_suite_to_hdfs()`: Persistence
* **How**: It converts the final Pandas DataFrame back into a distributed Spark DataFrame and writes it as a **Parquet** file.
* **Why**: Parquet is a columnar storage format that is highly optimized for the EPFL Hadoop cluster. Saving here ensures that our evaluation scripts can load the benchmark instantly without re-running the hours-long extraction process.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd

# ==========================================
# FUNCTION 1: GENERATE THE TEST SUITE
# ==========================================
def generate_test_suite(full_data, test_days, major_hubs, samples_per_level=50):
    """
    Extracts, joins, categorizes, and samples journeys from 0 to 3 transfers.
    """
    print(f"⏳ [Func 1] Filtering data for {len(test_days)} days...")
    
    # 1. Filter Data & Assign Sequences
    day_data = full_data.filter(F.col("operating_day").isin(test_days))
    trip_win = Window.partitionBy("trip_id").orderBy("arr_time")
    day_data = day_data.withColumn("seq", F.row_number().over(trip_win))
    day_data = day_data.withColumn("max_seq", F.max("seq").over(Window.partitionBy("trip_id")))

    # 2. Extract Direct Journeys (0 Transfers)
    direct = day_data.filter(F.col("seq") == 1).alias("o") \
        .join(day_data.filter(F.col("seq") == F.col("max_seq")).alias("d"), on="trip_id", how="inner") \
        .filter(F.col("o.stop_name") != F.col("d.stop_name")) \
        .select(
            F.col("o.operating_day").alias("operating_day"),
            F.col("o.bpuic").alias("origin_stop_id"),   # <--- ADDED ID
            F.col("d.bpuic").alias("dest_stop_id"),     # <--- ADDED ID
            F.col("o.stop_name").alias("origin_stop"),
            F.col("d.stop_name").alias("dest_stop"),
            F.lit("None").alias("transfer_hub"),
            F.lit(0).alias("num_transfers"),
            F.col("o.dep_time").alias("departure_time"),
            F.col("d.arr_time").alias("target_T"),
            F.col("d.actual_delay_min").alias("final_delay_min")
        ).dropDuplicates(["operating_day", "origin_stop_id", "dest_stop_id", "departure_time"])

    # 3. Define the Legs for Transfers
    leg1 = day_data.filter(F.col("seq") == 1).alias("l1_o") \
        .join(day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l1_h"), on="trip_id") \
        .select(F.col("trip_id").alias("t1"), F.col("l1_o.operating_day").alias("op_day"), 
                F.col("l1_o.bpuic").alias("origin_id"), F.col("l1_o.stop_name").alias("origin"), # <--- ADDED ID
                F.col("l1_o.dep_time").alias("t1_dep"), F.col("l1_h.stop_name").alias("hub1"), F.col("l1_h.arr_time").alias("t1_arr"))

    leg2 = day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l2_h1") \
        .join(day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l2_h2"), on="trip_id") \
        .filter(F.col("l2_h1.seq") < F.col("l2_h2.seq")) \
        .select(F.col("trip_id").alias("t2"), F.col("l2_h1.stop_name").alias("hub1"), F.col("l2_h1.dep_time").alias("t2_dep"), F.col("l2_h2.stop_name").alias("hub2"), F.col("l2_h2.arr_time").alias("t2_arr"))

    leg3 = day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l3_h2") \
        .join(day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l3_h3"), on="trip_id") \
        .filter(F.col("l3_h2.seq") < F.col("l3_h3.seq")) \
        .select(F.col("trip_id").alias("t3"), F.col("l3_h2.stop_name").alias("hub2"), F.col("l3_h2.dep_time").alias("t3_dep"), F.col("l3_h3.stop_name").alias("hub3"), F.col("l3_h3.arr_time").alias("t3_arr"))

    leg4 = day_data.filter((F.col("seq") > 1) & (F.col("stop_name").isin(major_hubs))).alias("l4_h3") \
        .join(day_data.filter(F.col("seq") == F.col("max_seq")).alias("l4_d"), on="trip_id") \
        .select(F.col("trip_id").alias("t4"), F.col("l4_h3.stop_name").alias("hub3"), F.col("l4_h3.dep_time").alias("t4_dep"), 
                F.col("l4_d.bpuic").alias("dest_id"), F.col("l4_d.stop_name").alias("dest"), # <--- ADDED ID
                F.col("l4_d.arr_time").alias("target_T"), F.col("l4_d.actual_delay_min").alias("final_delay_min"))

    print("⏳ [Func 1] Computing 1-Transfer Journeys...")
    trans1 = leg1.join(leg4.withColumnRenamed("hub3", "hub1").withColumnRenamed("t4", "t2").withColumnRenamed("t4_dep", "t2_dep"), on="hub1", how="inner") \
        .filter((F.col("t1") != F.col("t2")) & (F.col("t2_dep") > F.col("t1_arr")) & (F.col("t2_dep") <= F.col("t1_arr") + F.expr("INTERVAL 30 MINUTES")) & (F.col("t2_dep") >= F.col("t1_arr") + F.expr("INTERVAL 5 MINUTES")) & (F.col("origin") != F.col("dest"))) \
        .select(F.col("op_day").alias("operating_day"), 
                F.col("origin_id").alias("origin_stop_id"), F.col("dest_id").alias("dest_stop_id"), # <--- ADDED ID
                F.col("origin").alias("origin_stop"), F.col("dest").alias("dest_stop"), 
                F.col("hub1").alias("transfer_hub"), F.lit(1).alias("num_transfers"), F.col("t1_dep").alias("departure_time"), F.col("target_T"), F.col("final_delay_min")).dropDuplicates(["operating_day", "origin_stop_id", "dest_stop_id", "departure_time"])

    print("⏳ [Func 1] Computing 2-Transfer Journeys...")
    t1_to_t2 = leg1.join(leg2, on="hub1", how="inner").filter((F.col("t1") != F.col("t2")) & (F.col("t2_dep") > F.col("t1_arr")) & (F.col("t2_dep") <= F.col("t1_arr") + F.expr("INTERVAL 30 MINUTES")) & (F.col("t2_dep") >= F.col("t1_arr") + F.expr("INTERVAL 5 MINUTES")))
    trans2 = t1_to_t2.join(leg4.withColumnRenamed("hub3", "hub2").withColumnRenamed("t4", "t3").withColumnRenamed("t4_dep", "t3_dep"), on="hub2", how="inner") \
        .filter((F.col("t2") != F.col("t3")) & (F.col("t1") != F.col("t3")) & (F.col("t3_dep") > F.col("t2_arr")) & (F.col("t3_dep") <= F.col("t2_arr") + F.expr("INTERVAL 30 MINUTES")) & (F.col("t3_dep") >= F.col("t2_arr") + F.expr("INTERVAL 5 MINUTES")) & (F.col("origin") != F.col("dest"))) \
        .select(F.col("op_day").alias("operating_day"), 
                F.col("origin_id").alias("origin_stop_id"), F.col("dest_id").alias("dest_stop_id"), # <--- ADDED ID
                F.col("origin").alias("origin_stop"), F.col("dest").alias("dest_stop"), 
                F.concat_ws(" -> ", F.col("hub1"), F.col("hub2")).alias("transfer_hub"), F.lit(2).alias("num_transfers"), F.col("t1_dep").alias("departure_time"), F.col("target_T"), F.col("final_delay_min")).dropDuplicates(["operating_day", "origin_stop_id", "dest_stop_id", "departure_time"])

    print("⏳ [Func 1] Computing 3-Transfer Journeys...")
    t2_to_t3 = t1_to_t2.join(leg3, on="hub2", how="inner").filter((F.col("t2") != F.col("t3")) & (F.col("t1") != F.col("t3")) & (F.col("t3_dep") > F.col("t2_arr")) & (F.col("t3_dep") <= F.col("t2_arr") + F.expr("INTERVAL 30 MINUTES")) & (F.col("t3_dep") >= F.col("t2_arr") + F.expr("INTERVAL 5 MINUTES")))
    trans3 = t2_to_t3.join(leg4, on="hub3", how="inner") \
        .filter((F.col("t3") != F.col("t4")) & (F.col("t2") != F.col("t4")) & (F.col("t1") != F.col("t4")) & (F.col("t4_dep") > F.col("t3_arr")) & (F.col("t4_dep") <= F.col("t3_arr") + F.expr("INTERVAL 30 MINUTES")) & (F.col("t4_dep") >= F.col("t3_arr") + F.expr("INTERVAL 5 MINUTES")) & (F.col("origin") != F.col("dest"))) \
        .select(F.col("op_day").alias("operating_day"), 
                F.col("origin_id").alias("origin_stop_id"), F.col("dest_id").alias("dest_stop_id"), # <--- ADDED ID
                F.col("origin").alias("origin_stop"), F.col("dest").alias("dest_stop"), 
                F.concat_ws(" -> ", F.col("hub1"), F.col("hub2"), F.col("hub3")).alias("transfer_hub"), F.lit(3).alias("num_transfers"), F.col("t1_dep").alias("departure_time"), F.col("target_T"), F.col("final_delay_min")).dropDuplicates(["operating_day", "origin_stop_id", "dest_stop_id", "departure_time"])

    # 4. Union & Categorize
    print("⏳ [Func 1] Executing Spark plan and converting to Pandas...")
    ultimate_pool = direct.unionByName(trans1).unionByName(trans2).unionByName(trans3)
    queries_pdf = ultimate_pool.toPandas()

    def categorize_level(row):
        t, d = row['num_transfers'], row['final_delay_min']
        if t == 0 and d <= 3: return "Level 1"
        elif t == 0 and d > 3:  return "Level 2"
        elif t == 1 and d <= 3: return "Level 3"
        elif t == 1 and d > 3:  return "Level 4"
        elif t == 2 and d <= 3: return "Level 5"
        elif t == 2 and d > 3:  return "Level 6"
        elif t == 3 and d <= 3: return "Level 7"
        elif t == 3 and d > 3:  return "Level 8"
        return "Other"

    queries_pdf['Test_Category'] = queries_pdf.apply(categorize_level, axis=1)
    queries_pdf = queries_pdf[queries_pdf['Test_Category'] != "Other"]

    # 5. Stratified Sampling
    actual_min_available = queries_pdf['Test_Category'].value_counts().min()
    final_N = min(samples_per_level, actual_min_available)
    
    final_suite = queries_pdf.groupby('Test_Category').sample(n=final_N, random_state=42).reset_index(drop=True)
    final_suite = final_suite.sort_values(by=["num_transfers", "final_delay_min"], ascending=[True, False]).reset_index(drop=True)
    
    print("✅ [Func 1] Test Suite Generation Complete!")
    return final_suite


# ==========================================
# FUNCTION 2: DISPLAY STATS & PREVIEW
# ==========================================
def display_test_suite_stats(queries_pdf):
    """
    Displays the distribution of the generated categories and previews the data.
    """
    print("📊 [Func 2] Distribution of Test Queries:")
    display(queries_pdf['Test_Category'].value_counts().sort_index().to_frame(name="Count"))
    
    print("\n👀 [Func 2] Preview of the Test Suite (Top 2 from each level):")
    display(queries_pdf.groupby('Test_Category').head(2).sort_values(by="Test_Category"))


# ==========================================
# FUNCTION 3: SAVE TO HDFS
# ==========================================
def save_test_suite_to_hdfs(queries_pdf, spark_session, hdfs_path):
    """
    Converts the Pandas DataFrame back to Spark and saves it securely to Hadoop.
    """
    print(f"⏳ [Func 3] Saving dataset to HDFS at: {hdfs_path}")
    final_val_spark = spark_session.createDataFrame(queries_pdf)
    final_val_spark.write.format("parquet").mode("overwrite").save(hdfs_path)
    print("✅ [Func 3] Successfully saved to Hadoop!")

In [ ]:
import pandas as pd

# 1. Define your categorized testing days
testing_calendar = {
    "Regular Days": ["2025-11-12", "2025-12-10", "2026-01-20"],
    "Weekend Days": ["2025-11-15", "2025-12-14", "2026-01-24"],
    "Special Days": ["2025-12-25", "2025-12-31", "2026-01-02"]
}

major_hubs = [
    "Zürich HB", "Bern", "Lausanne", "Genève", "Basel SBB", 
    "Luzern", "Winterthur", "Biel/Bienne", "St. Gallen", 
    "Fribourg/Freiburg", "Olten", "Zug", 
    "Lugano", "Bellinzona", "Chur", "Sion", "Neuchâtel", "Aarau"
]
save_path = f"{hadoopFS}/user/{username}/master_e2e_benchmark_v2.parquet"

print(f"⏳ Building the Ultimate E2E Benchmark across 3 Months...")

# 2. Loop through the categories and days
for category, days in testing_calendar.items():
    print(f"\n======================================")
    print(f"🚂 Processing Category: {category}")
    print(f"======================================")
    
    for single_day in days:
        print(f"\n👉 Running Data Extraction for: {single_day}...")
        
        try:
            # We use samples_per_level=50 to get 400 highly balanced routes per day
            daily_suite = generate_test_suite(full_data, [single_day], major_hubs, samples_per_level=50) # CAN CHANGE THIS VALUE TO HAVE MORE SAMPLES !!!!!!!!!
            
            # Add a column so you know WHICH type of day this was in your final analysis!
            daily_suite['Day_Type'] = category
            
            # Convert to Spark
            daily_spark_df = spark.createDataFrame(daily_suite)
            
            # APPEND safely to Hadoop
            daily_spark_df.write.format("parquet").mode("append").save(save_path)
            
            print(f"✅ Safely appended {len(daily_suite)} journeys for {single_day} to HDFS.")
            
        except Exception as e:
            print(f"⚠️ Error on {single_day}: {e}")

print("\n🎉 MASTER BENCHMARK COMPLETE!")
print(f"Your multi-month, fully categorized dataset is saved at: {save_path}")

In [5]:

from pyspark.sql.window import Window
import pyspark.sql.functions as F
save_path = f"{hadoopFS}/user/{username}/master_e2e_benchmark_v2.parquet"
benchmark_df = spark.read.parquet(save_path)

total_rows = benchmark_df.count()
print(f"📊 Total journeys stored in Master Benchmark: {total_rows}")

print("\n📂 Data Schema:")
benchmark_df.printSchema()

print("\n👀 Stratified Preview (10 journeys per Level):")

preview_window = Window.partitionBy("Test_Category").orderBy("operating_day", "departure_time")

stratified_preview = benchmark_df.withColumn("rank", F.row_number().over(preview_window)) \
                                 .filter(F.col("rank") <= 10) \
                                 .drop("rank")

preview_pd = stratified_preview.toPandas().sort_values("Test_Category")

display(preview_pd)

📊 Total journeys stored in Master Benchmark: 3600

📂 Data Schema:
root
 |-- operating_day: date (nullable = true)
 |-- origin_stop_id: long (nullable = true)
 |-- dest_stop_id: long (nullable = true)
 |-- origin_stop: string (nullable = true)
 |-- dest_stop: string (nullable = true)
 |-- transfer_hub: string (nullable = true)
 |-- num_transfers: long (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- target_T: timestamp (nullable = true)
 |-- final_delay_min: double (nullable = true)
 |-- Test_Category: string (nullable = true)
 |-- Day_Type: string (nullable = true)


👀 Stratified Preview (10 journeys per Level):


,operating_day,origin_stop_id,dest_stop_id,origin_stop,dest_stop,transfer_hub,num_transfers,departure_time,target_T,final_delay_min,Test_Category,Day_Type
0,2025-11-12,8595511,8590618,"Dietikon, Heimstrasse","Geroldswil, Zentrum",None,0,2025-11-12 05:19:00,2025-11-12 05:25:00,0.000000,Level 1,Regular Days
1,2025-11-12,8589006,8582262,"Bern, Wittigkofen","Bern Brünnen Westside, Bahnhof",None,0,2025-11-12 05:44:00,2025-11-12 06:19:00,1.650000,Level 1,Regular Days
2,2025-11-12,8516161,8508207,Bern Wankdorf,Langnau i.E.,None,0,2025-11-12 05:46:00,2025-11-12 06:23:00,1.866667,Level 1,Regular Days
3,2025-11-12,8591676,8573602,"Wil SG, Bienenstrasse","Wil SG, Bahnhof",None,0,2025-11-12 06:03:00,2025-11-12 06:13:00,0.000000,Level 1,Regular Days
4,2025-11-12,8591046,8591122,"Zürich, Aspholz","Zürich, ETH Hönggerberg",None,0,2025-11-12 06:15:00,2025-11-12 06:24:00,1.266667,Level 1,Regular Days
...,...,...,...,...,...,...,...,...,...,...,...,...
74,2025-11-12,8502010,8500113,St. Erhard-Knutwil,Laufen,Olten -> Bern -> Basel SBB,3,2025-11-12 05:53:00,2025-11-12 08:31:00,4.150000,Level 8,Regular Days
75,2025-11-12,8507092,8515993,Belp Steinbach,Baar Lindenpark,Biel/Bienne -> Winterthur -> Zug,3,2025-11-12 06:10:00,2025-11-12 10:56:00,12.083333,Level 8,Regular Days
76,2025-11-12,8504181,8515993,Givisiez,Baar Lindenpark,Neuchâtel -> Winterthur -> Zug,3,2025-11-12 06:21:00,2025-11-12 11:26:00,10.483333,Level 8,Regular Days
77,2025-11-12,8501605,8507000,Visp,Bern,Sion -> Lausanne -> Bern,3,2025-11-12 07:09:00,2025-11-12 10:25:00,23.016667,Level 8,Regular Days


# 🌍 Proof of Multimodality: Analyzing the Transport Ecosystem

To ensure our **Connection Scan Algorithm (CSA)** is robust, it must handle the full complexity of Swiss public transport. We performed a forensic analysis of our dataset to verify that it includes not only major SBB trains but also the "Last Mile" network (Buses, Trams, and Regional Rails).

---

## 🔍 Validation Results

Our analysis utilized three distinct "detective" methods to prove the diversity of the data:

### 1. The "Comma & Keyword" Check (Naming Patterns)
In the Swiss transport naming convention (**DIDOK**), stop names provide clear clues about the transport mode:
* **Keywords found:** Our search identified terms like **"Dorf"** (Village Center), **"Post"** (PostBus stops), and **"Place"** (City Squares). 
* **The Comma Factor:** Roughly **16% of our stops** explicitly use the comma format (e.g., *Aigle, Place-du-Marché*). While many bus stops share names with train stations, these unique identifiers prove we are capturing stops located in the heart of cities and villages where InterCity trains cannot go.

### 2. The "Operator Soup" (Diversity of Companies)
Our dataset contains a wide variety of transport operators beyond the national SBB/CFF/FFS. The presence of these abbreviations confirms a multimodal environment:
* **Bus-Specific Operators:** `BSU` (Solothurn), `VMCV` (Montreux/Vevey), and `RVBW` (Baden).
* **Regional Rail & Hybrid:** `LEB` (Lausanne-Échallens-Bercher) and `ASM Auto` (Aare Seeland mobil).
* **Mountain/Specialty Lines:** `WAB` (Wengernalpbahn - cogwheel trains).

### 3. Geographical Depth
The data spans from major international hubs like **Zürich Flughafen** and **Basel SBB** to micro-locations like **Rhäzüns, Dorfplatz** and **Itschnach, Dorf**. This range ensures that our benchmark tests the algorithm's ability to navigate from a global gateway down to a local neighborhood.

---

## 💡 Why This Matters for our Project

A routing algorithm is only as good as the data it navigates. By proving our dataset is **Multimodal**, we confirm that our **End-to-End Benchmark** reflects real-world travel:
1.  **Complexity:** A 3-transfer journey in our test suite likely involves a mix of local buses and national trains.
2.  **Reliability:** Our ML model is being tested on its ability to predict delays for diverse vehicles—from a bus stuck in city traffic to a train delayed on the tracks.
3.  **Realism:** We are solving the "Door-to-Door" problem, not just the "Station-to-Station" problem.

In [6]:
from pyspark.sql import functions as F

# Count stops with commas vs. stops without
multimodal_check = benchmark_df.withColumn(
    "is_likely_bus_tram", F.col("origin_stop").contains(",")
).groupBy("is_likely_bus_tram").count()

display(multimodal_check.toPandas())

,is_likely_bus_tram,count
0,False,3025
1,True,575


In [7]:
# See all the different companies in your data
operators = full_data.select("operator_abrv").distinct().limit(20)
display(operators.toPandas())

,operator_abrv
0,NeTS-ÖBB
1,LEB
2,TPC
3,ARL
4,WAB
5,BSU
6,BS
7,RVBW
8,RA
9,AAGU


In [8]:
# Search for specific transport keywords
keywords = ["Post", "Zentrum", "Dorf", "Place", "Kante", "Platform"]
pattern = "|".join(keywords)

non_train_stops = benchmark_df.filter(F.col("origin_stop").rlike(pattern)).select("origin_stop").distinct().limit(10)
display(non_train_stops.toPandas())

,origin_stop
0,"Ganterschwil, alte Post"
1,"Gähwil, Dorf"
2,"Chur, Post 1"
3,"Wyssachen, Dorf"
4,Grand-Saconnex-Place
5,"Bignasco, Posta"
6,Riehen Dorf
7,Schleitheim Poststrasse
8,"Humlikon, Dorfplatz"
9,"Lamone-Cadempino, Posta/Staz."
